<div align="center">

# 🌾 agrometeorologiapy

### Agrometeorology formulas in Python — a guided tour, function by function

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologiapy/blob/main/examples/tutorial_colab_en.ipynb)
[![PyPI](https://img.shields.io/pypi/v/agrometeorologiapy.svg)](https://pypi.org/project/agrometeorologiapy/)
[![License: BSD-3-Clause](https://img.shields.io/badge/license-BSD--3--Clause-blue.svg)](https://github.com/fcoliveira-utfpr/agrometeorologiapy/blob/main/LICENSE)

*[🇧🇷 Versão em português](https://colab.research.google.com/github/fcoliveira-utfpr/agrometeorologiapy/blob/main/examples/tutorial_colab.ipynb)*

</div>

---

This notebook installs **`agrometeorologiapy`** from PyPI and walks through
**every one of its functions** using the English API, `agrometeorologiapy.en`,
with a short practical problem and a runnable numerical example for each.

Terms and symbols follow FAO Irrigation and Drainage Paper 56 (Allen et al.,
1998) wherever they exist: $R_a$ (extraterrestrial radiation), $R_s$ (solar
radiation), $\omega_s$ (sunset hour angle), $e_s$, $e_a$, $u_2$, $ET_o$...

The whole notebook follows a single fictional scenario — **Santa Rita Farm**,
in Ponta Grossa, Paraná, southern Brazil — so that the results of one section
feed the next, as in a real agrometeorological analysis.

> 💡 **How to use:** click *"Open in Colab"* above, then **Runtime → Run all**
> (or run cell by cell with `Shift+Enter`). Nothing needs to be installed on
> your computer.

## 🔧 Installation

Installing the latest release from PyPI:

In [ ]:
!pip install -q agrometeorologiapy

In [ ]:
import agrometeorologiapy
from agrometeorologiapy import en as agm
import pandas as pd

print("agrometeorologiapy", agrometeorologiapy.__version__, "installed ✅")

agrometeorologiapy 0.3.0 installed ✅


> ℹ️ **Two APIs, one implementation.** `agrometeorologiapy.en` is the core of
> the package. The same functions are also available with Portuguese names at
> the top level (`import agrometeorologiapy as amp`), as used in Brazilian
> agrometeorology courses — for example `agm.extraterrestrial_radiation` is
> `amp.irradiancia_extraterrestre`. Both give identical results. The full name
> mapping is in
> [docs/API_PT_EN.md](https://github.com/fcoliveira-utfpr/agrometeorologiapy/blob/main/docs/API_PT_EN.md).

## 📍 Reference scenario

**Santa Rita Farm**, Ponta Grossa region, Paraná, Brazil.

| Variable | Value | Meaning |
|---|---|---|
| `lat` | -25.09° | Latitude (negative = southern hemisphere) |
| `z` | 875 m | Elevation of the weather station |
| `day`, `month` | 15, 9 | Reference date (September 15) |
| `hour`, `minute` | 14, 20 | Time of the observation |
| `pole_height` | 4 m | Height of a pole, for the shadow example |
| `Tmax`, `Tmin` | 27.5 °C, 14.2 °C | Daily maximum and minimum temperature |
| `RH` | 58 % | Relative humidity |
| `uz`, `z_anemometer` | 2.4 m/s, 10 m | Wind speed measured by an anemometer at 10 m |

These variables are used throughout the notebook — run the cell below once
and move on.

In [ ]:
# Reference station: Santa Rita Farm, Ponta Grossa - PR, Brazil
lat = -25.09
z = 875
day, month = 15, 9
hour, minute = 14, 20
pole_height = 4
Tmax, Tmin = 27.5, 14.2
RH = 58
uz, z_anemometer = 2.4, 10

---
## 1️⃣ Trigonometry helper functions

Agrometeorological formulas work with angles in **degrees**, not radians.
`sind`, `cosd`, `tand` and `acosd` are shortcuts used internally by almost
every other function in the package — and are also available to you.

> 📌 **Problem:** quickly check the sine, cosine and tangent of a 37° angle,
> and the arc-cosine of 0.6 (in degrees, without worrying about radians).

In [ ]:
angle = 37
print("sind(37°) =", agm.sind(angle))
print("cosd(37°) =", agm.cosd(angle))
print("tand(37°) =", agm.tand(angle))
print("acosd(0.6) =", agm.acosd(0.6), "degrees")

sind(37°) = 0.6018150231520483
cosd(37°) = 0.7986355100472928
tand(37°) = 0.7535540501027942
acosd(0.6) = 53.13010235415599 degrees


---
## 2️⃣ Solar Radiation

In this section we compute, step by step, the position of the Sun and the
radiation reaching Santa Rita Farm on September 15. Each function uses the
result of the previous one — the usual way solar radiation is computed in
practice (FAO-56, chapter 3).

### 2.1 `day_of_year` — Day of the year (J)

> 📌 First of all, which day of the year is September 15? It is the input of
> the solar declination formula.

In [ ]:
J = agm.day_of_year(day, month)
print(f"Day of the year (J): {J}")

Day of the year (J): 258


### 2.2 `solar_declination` — Solar declination (δ)

> 📌 With J, compute the solar declination of the day — how far the Sun is
> from the celestial equator at this time of the year.

In [ ]:
decl = agm.solar_declination(J)
print(f"Solar declination (δ): {decl:.4f}°")

Solar declination (δ): 1.8147°


### 2.3 `hour_angle` — Hour angle (ω)

> 📌 The observation was made at 14:20. Convert this local (solar) time into
> an hour angle (0° at solar noon).

In [ ]:
w = agm.hour_angle(hour, minute)
print(f"Hour angle (ω): {w:.2f}°")

Hour angle (ω): 35.00°


### 2.4 `solar_zenith_angle` — Zenith angle at solar noon

> 📌 What is the smallest zenith angle of the Sun on this day, at the
> latitude of the farm?

In [ ]:
Z = agm.solar_zenith_angle(lat, decl)
print(f"Zenith angle (Z): {Z:.2f}°")

Zenith angle (Z): 26.90°


### 2.5 `solar_azimuth` — Solar azimuth

> 📌 In which direction (relative to the north-south line) is the Sun?

In [ ]:
alpha = agm.solar_azimuth(lat, decl, Z)
print(f"Solar azimuth (α): {alpha:.2f}°")

Solar azimuth (α): 180.00°


### 2.6 `shadow_length` — Shadow length

> 📌 There is a 4 m pole in the farmyard. How long is its shadow?

In [ ]:
S = agm.shadow_length(pole_height, Z)
print(f"Shadow length (S): {S:.2f} m")

Shadow length (S): 2.03 m


### 2.7 `sunset_hour_angle` — Sunset hour angle (ωs)

> 📌 At which hour angle does the Sun set, at this latitude and time of the
> year? (By symmetry, it is also the sunrise hour angle in absolute value.)

In [ ]:
ws = agm.sunset_hour_angle(lat, decl)
print(f"Sunset hour angle (ωs): {ws:.2f}°")

Sunset hour angle (ωs): 89.15°


### 2.8 `daylight_hours` — Daylight hours (N)

> 📌 How many hours of daylight does this day have, from sunrise to sunset?

In [ ]:
N = agm.daylight_hours(ws)
print(f"Daylight hours (N): {N:.2f} hours")

Daylight hours (N): 11.89 hours


### 2.9 `inverse_relative_distance` — Inverse relative distance Earth-Sun (dr)

> 📌 The Earth is not always at the same distance from the Sun. What is the
> correction for this day of the year?

In [ ]:
dr = agm.inverse_relative_distance(J)
print(f"Inverse relative distance Earth-Sun (dr): {dr:.4f}")

Inverse relative distance Earth-Sun (dr): 0.9912


### 2.10 `extraterrestrial_radiation` — Radiation at the top of the atmosphere (Ra)

> 📌 Before any attenuation by the atmosphere, how much solar energy reaches
> the top of the atmosphere above the farm on this day?

In [ ]:
Ra = agm.extraterrestrial_radiation(lat, decl, ws, dr)
print(f"Extraterrestrial radiation (Ra): {Ra:.2f} MJ m-2 day-1")

Extraterrestrial radiation (Ra): 32.95 MJ m-2 day-1


### 2.11 `sunshine_duration` — Estimated sunshine duration (n)

> 📌 There is no sunshine recorder at the farm. Estimate how many hours of
> bright sunshine (n) the day had from the daily temperature range.

In [ ]:
n = agm.sunshine_duration(N, Tmax, Tmin, lat)
print(f"Estimated sunshine duration (n): {n:.2f} hours")

Estimated sunshine duration (n): 9.84 hours


### 2.12 `solar_radiation_angstrom` — Solar radiation by the Angstrom formula (Rs)

> 📌 Combine the sunshine duration with Ra to obtain the solar radiation that
> actually reaches the surface (FAO-56 eq. 35).

In [ ]:
Rs_ang = agm.solar_radiation_angstrom(n, N, Ra, lat)
print(f"Solar radiation — Angstrom (Rs): {Rs_ang:.2f} MJ m-2 day-1")

Solar radiation — Angstrom (Rs): 22.83 MJ m-2 day-1


### 2.13 `solar_radiation_hargreaves` — Solar radiation by Hargreaves' formula (Rs)

> 📌 And without any sunshine data? Hargreaves' radiation formula (FAO-56
> eq. 50) only needs Tmax, Tmin and Ra.

In [ ]:
Rs = agm.solar_radiation_hargreaves(Tmax, Tmin, Ra)
print(f"Solar radiation — Hargreaves (Rs): {Rs:.2f} MJ m-2 day-1")

Solar radiation — Hargreaves (Rs): 19.23 MJ m-2 day-1


### 2.14 `relative_sunshine_duration` — Relative sunshine duration (n/N)

> 📌 The same Angstrom formula, solved for n/N: when the station measures (or
> you estimated) solar radiation but there is no sunshine recorder, what
> fraction of the day had bright sunshine?

In [ ]:
nN = agm.relative_sunshine_duration(Rs, Ra, lat)
print(f"n/N from the Hargreaves Rs: {nN:.3f}  (n ≈ {nN * N:.1f} h of bright sunshine)")

# Check: inverting the Angstrom Rs gives back the n/N used to compute it
print(f"n/N from the Angstrom Rs: {agm.relative_sunshine_duration(Rs_ang, Ra, lat):.3f}  |  n/N = {n / N:.3f}")

n/N from the Hargreaves Rs: 0.617  (n ≈ 7.3 h of bright sunshine)
n/N from the Angstrom Rs: 0.827  |  n/N = 0.827


---
## 3️⃣ Temperature

### 3.1 `mean_temperature` — Mean daily temperature (Tmean)

> 📌 The simplest and most common method: the mean of Tmax and Tmin
> (FAO-56 eq. 9).

In [ ]:
Tmean = agm.mean_temperature(Tmax, Tmin)
print(f"Mean daily temperature (Tmean): {Tmean:.2f} °C")

Mean daily temperature (Tmean): 20.85 °C


### 3.2 `mean_temperature_observations` — Mean of station observations

> 📌 The farm's automatic weather station logged 6 readings during the day.
> What is the mean temperature of the period?

In [ ]:
hourly_temperatures = [16.8, 19.4, 23.9, 27.1, 26.3, 20.6]
Tmean_obs = agm.mean_temperature_observations(hourly_temperatures)
print(f"Mean temperature (station observations): {Tmean_obs:.2f} °C")

Mean temperature (station observations): 22.35 °C


---
## 4️⃣ Air Humidity

With the daily temperatures (`Tmax`, `Tmin`) and the relative humidity
(`RH = 58%`) of the scenario, we compute the whole chain of humidity
variables.

### 4.1 `saturation_vapour_pressure` — Saturation vapour pressure (es)

> 📌 Saturation vapour pressure is not linear in temperature, so FAO-56
> computes it at Tmax and Tmin and uses the **mean of the two** as the daily
> `es`, rather than applying the Tetens equation to Tmean.

In [ ]:
es_tmax = agm.saturation_vapour_pressure(Tmax)
es_tmin = agm.saturation_vapour_pressure(Tmin)
es = (es_tmax + es_tmin) / 2

print(f"Saturation vapour pressure at Tmax: {es_tmax:.4f} kPa")
print(f"Saturation vapour pressure at Tmin: {es_tmin:.4f} kPa")
print(f"Mean saturation vapour pressure (es): {es:.4f} kPa")

Saturation vapour pressure at Tmax: 3.6710 kPa
Saturation vapour pressure at Tmin: 1.6194 kPa
Mean saturation vapour pressure (es): 2.6452 kPa


### 4.2 `actual_vapour_pressure` — Actual vapour pressure (ea)

> 📌 With RH = 58%, what is the vapour pressure actually in the air?

In [ ]:
ea = agm.actual_vapour_pressure(es, RH)
print(f"Actual vapour pressure (ea): {ea:.4f} kPa")

Actual vapour pressure (ea): 1.5342 kPa


### 4.3 `vapour_pressure_deficit` — Vapour pressure deficit (es − ea)

> 📌 How "far" is the air from saturation? The deficit is proportional to
> the evaporative demand of the atmosphere.

In [ ]:
vpd = agm.vapour_pressure_deficit(es, ea)
print(f"Vapour pressure deficit (es - ea): {vpd:.4f} kPa")

Vapour pressure deficit (es - ea): 1.1110 kPa


### 4.4 `atmospheric_pressure` — Atmospheric pressure from elevation

> 📌 The farm is at 875 m above sea level. What is the atmospheric pressure
> there (FAO-56 eq. 7)?

In [ ]:
Patm = agm.atmospheric_pressure(z)
print(f"Atmospheric pressure (P): {Patm:.4f} kPa")

Atmospheric pressure (P): 91.3757 kPa


### 4.5 `absolute_humidity` — Absolute humidity

> 📌 How many grams of water are actually in each cubic metre of air?

In [ ]:
AH = agm.absolute_humidity(ea, Tmean)
print(f"Absolute humidity: {AH:.2f} g/m³")

Absolute humidity: 11.31 g/m³


### 4.6 `saturation_humidity` — Saturation humidity

> 📌 And how many grams per cubic metre could the air hold at most, at the
> same temperature?

In [ ]:
SH = agm.saturation_humidity(es, Tmean)
print(f"Saturation humidity: {SH:.2f} g/m³")

Saturation humidity: 19.51 g/m³


### 4.7 `relative_humidity` — Relative humidity (check)

> 📌 Recomputing RH from `ea` and `es` — it must match the 58% used as input.

In [ ]:
RH_check = agm.relative_humidity(ea, es)
print(f"Recomputed relative humidity: {RH_check:.2f} %")

Recomputed relative humidity: 58.00 %


### 4.8 `dewpoint_temperature` — Dewpoint temperature (Tdew)

> 📌 To which temperature would the air have to cool, keeping its vapour
> content, for dew to start forming?

In [ ]:
Tdew = agm.dewpoint_temperature(ea)
print(f"Dewpoint temperature (Tdew): {Tdew:.2f} °C")

Dewpoint temperature (Tdew): 13.37 °C


### 4.9 `psychrometric_constant` — Psychrometric constant (γ)

> 📌 Essential in the combination methods of evapotranspiration
> (Priestley-Taylor, Penman-Monteith); it depends only on the atmospheric
> pressure (FAO-56 eq. 8).

In [ ]:
gamma = agm.psychrometric_constant(Patm)
print(f"Psychrometric constant (γ): {gamma:.6f} kPa/°C")

Psychrometric constant (γ): 0.060765 kPa/°C


---
## 5️⃣ Radiation Balance

> ℹ️ **Sign convention:** the package returns the net longwave radiation as a
> negative value (net loss) and computes $R_n = R_{ns} + R_{nl}$. FAO-56 writes
> $R_{nl}$ as a positive outgoing flux and $R_n = R_{ns} - R_{nl}$; the net
> radiation is the same.

### 5.1 `net_shortwave_radiation` — Net shortwave radiation (Rns)

> 📌 Of the incoming solar radiation (`Rs`, from section 2), how much is left
> after the surface reflection, with the albedo of the FAO-56 reference crop
> (0.23)?

In [ ]:
Rns = agm.net_shortwave_radiation(Rs, albedo=0.23)
print(f"Net shortwave radiation (Rns): {Rns:.2f} MJ m-2 day-1")

Net shortwave radiation (Rns): 14.81 MJ m-2 day-1


### 5.2 `net_longwave_radiation` — Net longwave radiation (Rnl)

> 📌 How much energy does the surface lose by emitting infrared radiation,
> corrected for cloudiness and air humidity (FAO-56 eq. 39)?

In [ ]:
Rso = (0.75 + 2e-5 * z) * Ra  # clear-sky solar radiation (FAO-56 eq. 37)
Rnl = agm.net_longwave_radiation(Tmax, Tmin, ea, Rs, Rso)
print(f"Clear-sky solar radiation (Rso): {Rso:.2f} MJ m-2 day-1")
print(f"Net longwave radiation (Rnl): {Rnl:.4f} MJ m-2 day-1")

Clear-sky solar radiation (Rso): 25.29 MJ m-2 day-1
Net longwave radiation (Rnl): -4.1401 MJ m-2 day-1


### 5.3 `net_radiation` — Net radiation (Rn)

> 📌 Adding the two balances, how much net energy is left at the surface for
> the physical and biological processes of the day?

In [ ]:
Rn = agm.net_radiation(Rns, Rnl)
print(f"Net radiation (Rn): {Rn:.2f} MJ m-2 day-1")

Net radiation (Rn): 10.67 MJ m-2 day-1


---
## 6️⃣ Evapotranspiration

First, two **monthly** methods (they need a series of 12 monthly mean
temperatures) — useful when only climate normals are available. Then the
**daily** methods, which reuse `Rn`, `gamma` and the other variables computed
so far.

### 6.1 `pet_thornthwaite` — Monthly PET by Thornthwaite (1948)

> 📌 With the historical monthly mean temperatures of Ponta Grossa, estimate
> the potential evapotranspiration (PET) of each month of the year.

In [ ]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
T_monthly = [22.1, 22.4, 21.6, 19.0, 15.8, 14.2, 13.9, 15.4, 17.1, 19.3, 20.7, 21.9]

df_monthly = pd.DataFrame({'month': months, 'T_mean_C': T_monthly})
df_thornthwaite = agm.pet_thornthwaite(df_monthly, lat=lat)
df_thornthwaite

,month,T_mean_C,PET_mm_month
0,Jan,22.1,108.90
1,Feb,22.4,96.83
2,Mar,21.6,94.63
3,Apr,19.0,66.96
4,May,15.8,45.65
5,Jun,14.2,34.85
6,Jul,13.9,35.01
7,Aug,15.4,44.90
8,Sep,17.1,56.72
9,Oct,19.3,78.84


### 6.2 `pet_camargo` — Monthly PET by Camargo (modified by Maluf)

> 📌 Compare with a second monthly method, which uses the extraterrestrial
> radiation instead of Thornthwaite's heat index.

In [ ]:
df_camargo = agm.pet_camargo(df_monthly, lat=lat)
df_camargo

,month,T_mean_C,Ra_MJ_m2day,Ra_mm_day,PET_mm_month
0,Jan,22.1,42.56,17.37,118.97
1,Feb,22.4,39.88,16.27,102.05
2,Mar,21.6,35.40,14.45,96.72
3,Apr,19.0,29.37,11.98,68.30
4,May,15.8,24.02,9.80,48.01
5,Jun,14.2,21.45,8.75,37.27
6,Jul,13.9,22.49,9.17,39.53
7,Aug,15.4,26.86,10.96,52.31
8,Sep,17.1,32.77,13.37,68.59
9,Oct,19.3,38.16,15.57,93.15


### 6.3 `slope_vapour_pressure_curve` — Slope of the saturation vapour pressure curve (Δ)

> 📌 Back to September 15: what is the slope of the Tetens curve at the mean
> temperature of the day? It is an input of the combination methods below
> (FAO-56 eq. 13).

In [ ]:
Delta = agm.slope_vapour_pressure_curve(Tmean)
print(f"Slope of the saturation vapour pressure curve (Δ): {Delta:.6f} kPa/°C")

Slope of the saturation vapour pressure curve (Δ): 0.151523 kPa/°C


### 6.4 `pet_hargreaves_samani` — PET by Hargreaves & Samani (1985)

> 📌 Estimate the daily PET from the temperature range and the
> extraterrestrial radiation only — useful when wind and humidity data are
> missing (FAO-56 eq. 52).

In [ ]:
PET_HS = agm.pet_hargreaves_samani(Ra, Tmax, Tmin, Tmean)
print(f"PET — Hargreaves-Samani: {PET_HS:.2f} mm/day")

PET — Hargreaves-Samani: 4.36 mm/day


### 6.5 `pet_priestley_taylor` — PET by Priestley & Taylor (1972)

> 📌 Now with the net radiation (`Rn`) and the psychrometric constant
> (`gamma`), apply the simplified Priestley-Taylor method (suited to
> well-watered surfaces).

In [ ]:
G = 0  # soil heat flux, negligible at the daily scale
PET_PT = agm.pet_priestley_taylor(Rn, G, Delta, gamma)
print(f"PET — Priestley-Taylor: {PET_PT:.2f} mm/day")

PET — Priestley-Taylor: 3.91 mm/day


### 6.6 `wind_speed_2m` — Wind speed at 2 m (u2)

> 📌 The farm's anemometer is 10 m above the ground, but Penman-Monteith needs
> the wind speed at 2 m. Convert it with the logarithmic wind profile (FAO-56
> eq. 47).

In [ ]:
u2 = agm.wind_speed_2m(uz, z_anemometer)
print(f"Wind speed at 2 m (u2): {u2:.2f} m/s")

Wind speed at 2 m (u2): 1.80 m/s


### 6.7 `eto_penman_monteith` — Reference evapotranspiration by FAO Penman-Monteith (ETo)

> 📌 The international standard method (FAO-56 eq. 6): combine radiation,
> wind (`u2`) and vapour pressure deficit to obtain the reference
> evapotranspiration.

In [ ]:
ETo = agm.eto_penman_monteith(Rn, G, Tmean, u2, es, ea, Delta, gamma)
print(f"ETo — FAO Penman-Monteith: {ETo:.2f} mm/day")

ETo — FAO Penman-Monteith: 4.13 mm/day


---
## 7️⃣ Growing Degree Days

New scenario: we are now the agronomist in charge of a **soybean** field
(base temperature Tb = 14 °C, thermal time requirement of the cycle
TT = 1350 °C·day), with the monthly mean temperatures forecast for the year.

### 7.1 `maturity_date` — When will the soybean reach maturity?

> 📌 The soybean was sown on October 15. Accumulating growing degree days
> month by month, on which date should it reach physiological maturity?

In [ ]:
month_numbers = list(range(1, 13))
Tmean_monthly = [23.1, 23.3, 22.4, 20.5, 17.2, 15.6, 15.3, 16.9, 18.6, 20.5, 21.7, 22.8]
Tmin_monthly = [18.9, 19.0, 17.8, 15.7, 12.3, 10.8, 10.2, 11.6, 13.4, 15.6, 16.9, 18.0]
Tmax_monthly = [27.6, 27.9, 27.3, 25.6, 22.4, 20.7, 20.7, 22.5, 24.1, 25.7, 26.8, 27.4]

Tb, TT = 14, 1350
sowing_day, sowing_month = 15, 10  # October 15

df_climate = pd.DataFrame({
    'day': [1] * 12,
    'month': month_numbers,
    'Tmean': Tmean_monthly,
    'Tmax': Tmax_monthly,
    'Tmin': Tmin_monthly,
})

maturity = agm.maturity_date(df_climate, Tb, TT, sowing_day, sowing_month, interval='M')
maturity

Sowing date: October 15
Physiological maturity date: March 23, 2024


,month,date,Tmean,GDD,GDD_period,GDD_cycle
0,Oct,2023-10-15,20.5,6.5,104.0,104.0
1,Nov,2023-11-01,21.7,7.7,231.0,335.0
2,Dec,2023-12-01,22.8,8.8,272.8,607.8
3,Jan,2024-01-01,23.1,9.1,282.1,889.9
4,Feb,2024-02-01,23.3,9.3,269.7,1159.6
5,Mar,2024-03-23,22.4,8.4,193.2,1352.8


### 7.2 `sowing_date` — When to sow to harvest on a target date?

> 📌 Now the inverse problem: you want to harvest around March 20 next year.
> On which date must the soybean be sown?

In [ ]:
maturity_day, maturity_month = 20, 3  # target harvest: March 20

sowing = agm.sowing_date(
    df_climate, Tb=10, TT=1350,
    maturity_day=maturity_day, maturity_month=maturity_month,
    interval='M',
)
sowing

Maturity date (reference): March 20
Required sowing date: December 5, 2022


,month,date,Tmean,GDD,GDD_period,GDD_cycle
0,Mar,2023-03-20,22.4,12.4,248.0,248.0
1,Feb,2023-02-01,23.3,13.3,372.4,620.4
2,Jan,2023-01-01,23.1,13.1,406.1,1026.5
3,Dec,2022-12-05,22.8,12.8,332.8,1359.3


---
## 8️⃣ Water Balance

### 8.1 `climatic_water_balance` — Annual climatic water balance

> 📌 With the mean monthly rainfall of the region and the monthly PET from
> section 6 (Thornthwaite), build the climatic water balance of the farm
> (Thornthwaite & Mather, 1955), with an available water capacity (AWC) of
> 125 mm.

Columns: soil water storage (STO), actual evapotranspiration (ETa), water
deficit (DEF) and water surplus (SUR).

In [ ]:
rainfall_monthly = [165.2, 148.7, 132.4, 88.6, 78.3, 118.5,
                    132.9, 95.4, 138.7, 158.2, 142.6, 172.3]

df_wb_input = pd.DataFrame({
    'month': months,
    'P (mm/month)': rainfall_monthly,
    'PET (mm/month)': df_thornthwaite['PET_mm_month'],
})

df_climatic_wb = agm.climatic_water_balance(df_wb_input, AWC=125.0)
df_climatic_wb[['month', 'P (mm/month)', 'PET (mm/month)', 'STO (mm/month)',
                'ETa (mm/month)', 'DEF (mm/month)', 'SUR (mm/month)']]

,month,P (mm/month),PET (mm/month),STO (mm/month),ETa (mm/month),DEF (mm/month),SUR (mm/month)
0,Jan,165.2,108.90,125.0,108.90,0.0,56.30
1,Feb,148.7,96.83,125.0,96.83,0.0,51.87
2,Mar,132.4,94.63,125.0,94.63,0.0,37.77
3,Apr,88.6,66.96,125.0,66.96,0.0,21.64
4,May,78.3,45.65,125.0,45.65,0.0,32.65
5,Jun,118.5,34.85,125.0,34.85,0.0,83.65
6,Jul,132.9,35.01,125.0,35.01,0.0,97.89
7,Aug,95.4,44.90,125.0,44.90,0.0,50.50
8,Sep,138.7,56.72,125.0,56.72,0.0,81.98
9,Oct,158.2,78.84,125.0,78.84,0.0,79.36


### 8.2 `crop_water_balance` — Crop water balance

> 📌 Zooming into the soybean cycle: with rainfall, the crop
> evapotranspiration (ETc = Kc × ETo) and an available water capacity that
> grows with the roots, month by month, compute the crop water balance — and
> the water requirement satisfaction index (WRSI = ETa/ETc).

In [ ]:
df_crop_input = pd.DataFrame({
    'P':   [52.3, 41.8, 28.6, 18.2, 65.4, 88.1],
    'ETc': [12.5, 22.8, 38.4, 51.2, 47.6, 30.1],
    'AWC': [18.0, 30.0, 42.0, 54.0, 54.0, 54.0],
})

df_crop_wb = agm.crop_water_balance(df_crop_input)
df_crop_wb

,P,ETc,AWC,P-ETc,STO,dSTO,ETa,DEF,SUR,WRSI
0,52.3,12.5,18.0,39.8,18.000000,0.000000,12.500000,0.000000,39.800000,1.000000
1,41.8,22.8,30.0,19.0,30.000000,12.000000,22.800000,0.000000,7.000000,1.000000
2,28.6,38.4,42.0,-9.8,23.756687,-6.243313,34.843313,3.556687,0.000000,0.907378
3,18.2,51.2,54.0,-33.0,12.893882,-10.862805,29.062805,22.137195,0.000000,0.567633
4,65.4,47.6,54.0,17.8,30.693882,17.800000,47.600000,0.000000,0.000000,1.000000
5,88.1,30.1,54.0,58.0,54.000000,23.306118,30.100000,0.000000,34.693882,1.000000


---
## 9️⃣ Climate Classification

With the monthly normals of Ponta Grossa used in sections 6 and 8
(`T_monthly`, `rainfall_monthly` and the Thornthwaite PET), we classify the
farm's climate by four methods. Each function has a version for **one
location** and a `_grid` version for **many locations at once** (e.g. every
pixel of a raster).

### 9.1 `koppen_classification` — Köppen-Geiger

> 📌 By the most widely used system (methodology of Alvares et al., 2013),
> what is the climate of the farm? It only needs temperature and rainfall.

In [ ]:
koppen = agm.koppen_classification(T_monthly, rainfall_monthly, lat)
print(f"Köppen-Geiger: {koppen['class']}")

Köppen-Geiger: Cfa


### 9.2 `thornthwaite_classification` — Thornthwaite (1948)

> 📌 Now from the water balance: humidity (Ih), aridity (Ia) and moisture
> (Im = Ih − 0.6 Ia) indices, thermal efficiency and summer concentration of
> PET, with the same AWC of 125 mm as section 8.1.

In [ ]:
th = agm.thornthwaite_classification(rainfall_monthly, df_thornthwaite['PET_mm_month'], lat, AWC=125)
print(f"Thornthwaite: {th['class']} — {th['description']}")
print(f"Ih = {th['Ih']:.1f}%   Ia = {th['Ia']:.1f}%   Im = {th['Im']:.1f}%")

Thornthwaite: B4rB'3a' — Humid, mesothermal
Ih = 82.1%   Ia = 0.0%   Im = 82.1%


### 9.3 `camargo_classification` — Camargo (1991) modified by Maluf (2000)

> 📌 A middle ground between Köppen and Thornthwaite: thermal class (annual
> and coldest-month temperature) + moisture class (annual deficit and
> surplus). The class codes are those of the original (Brazilian) system.

In [ ]:
cm = agm.camargo_classification(T_monthly, rainfall_monthly, df_thornthwaite['PET_mm_month'], lat, AWC=125)
print(f"Camargo: {cm['class']} — {cm['description']}")
print(f"Annual DEF = {cm['DEF_annual']:.1f} mm   Annual SUR = {cm['SUR_annual']:.1f} mm")

Camargo: ST-PU — Subtropical perhumid


Annual DEF = 0.0 mm   Annual SUR = 708.5 mm


### 9.4 `holdridge_classification` — Holdridge life zones

> 📌 Which vegetation formation would be expected in this climate? Holdridge
> combines the biotemperature with the ratio of PET to rainfall (38 life
> zones).

In [ ]:
hd = agm.holdridge_classification(T_monthly, rainfall_monthly, lat)
print(f"Holdridge: {hd['class']} (zone {hd['id']})")
print(f"Biotemperature = {hd['biotemperature']:.1f} °C   PET/P = {hd['PET_ratio']:.2f}")

Holdridge: Subtropical moist forest (zone 29)
Biotemperature = 18.6 °C   PET/P = 0.70


### 9.5 `_grid` versions — many locations (rasters)

> 📌 To map a region you cannot call the function pixel by pixel. The
> `_grid` versions take arrays with the 12 months on the first axis, shape
> `(12, ...)` — e.g. `(12, rows, columns)`, as read from a 12-band GeoTIFF
> with `rasterio` — and classify everything at once. Here, three "pixels":
> the farm, a site in the Amazon and one in the Brazilian semi-arid region.

In [ ]:
import numpy as np

T_grid = np.column_stack([
    T_monthly,
    [26.1, 26.0, 26.1, 26.2, 26.3, 26.4, 26.5, 27.1, 27.5, 27.4, 27.1, 26.6],  # Amazon
    [27.5, 27.2, 27.0, 26.5, 25.6, 24.6, 24.2, 24.9, 26.4, 27.8, 28.3, 28.0],  # semi-arid
])                                            # shape (12, 3)
P_grid = np.column_stack([
    rainfall_monthly,
    [264, 295, 315, 300, 255, 114, 87, 58, 83, 126, 183, 217],
    [95, 90, 130, 80, 25, 12, 8, 3, 5, 12, 50, 80],
])
lat_grid = np.array([lat, -3.1, -9.4])

print("Köppen:   ", agm.koppen_classification_grid(T_grid, P_grid, lat_grid)['class'])
print("Holdridge:", agm.holdridge_classification_grid(T_grid, P_grid, lat_grid)['class'])

Köppen:    ['Cfa' 'Am' 'BSh']
Holdridge: ['Subtropical moist forest' 'Tropical moist forest'
 'Tropical very dry forest']


---
## 🔟 Potential and attainable yield (FAO Agro-Ecological Zone model)

Last stop of the tour: how much soybean Santa Rita Farm **could** produce
with the weather of the season, and how much it **would** produce with the
rain that actually fell. The FAO Agro-Ecological Zone model (Doorenbos &
Kassam, 1979) does this at two levels:

```
 Ra, n/N, T ──► GP ──► × LAI_corr × CR × HI ÷ (1 − 0.01·moisture) ──► Yp
                                                                       │
 P, ETo, Kc, AWC ──► Crop water balance ──► ETa/ETc ──► × ky           ▼
                                                                       Yw ──► CE = Yw / Yp
```

Names follow the yield gap literature (van Ittersum et al., 2013): **Yp** is
the potential yield and **Yw** the attainable (water-limited) yield. Each
formula has its own function, and every constant or tabulated value is a
parameter with the usual value as default (e.g. `moisture=13`, `HI=0.35`,
`b=0.52`).

### 10.1 `relative_sunshine_duration` — n/N without a sunshine recorder

> 📌 The model splits the day into a clear-sky fraction (n/N) and an overcast
> fraction (1 − n/N). Without a sunshine recorder at the farm, n/N comes from
> `relative_sunshine_duration` (section 2.14), with `Rs` and `Ra` of
> September 15.

In [ ]:
nN = agm.relative_sunshine_duration(Rs, Ra, lat)   # a = 0.29 cos(lat), b = 0.52 (default)
print(f"Rs/Ra = {Rs / Ra:.3f}  →  n/N = {nN:.3f}")

Rs/Ra = 0.584  →  n/N = 0.617


### 10.2 `temperature_correction_overcast` and `temperature_correction_clear` — C3 or C4

> 📌 Photosynthetic efficiency depends on the carbon fixation pathway. Compare
> soybean (C3) and maize (C4) at several temperatures, for overcast (`cTo`)
> and clear (`cTc`) sky. The pathway is **required**: `'C3'` or `'C4'`.

In [ ]:
T_test = [12.5, 15.0, 20.0, 25.0, 30.0, 35.0]
pd.DataFrame({
    'T (°C)': T_test,
    'cTo C3': agm.temperature_correction_overcast(pd.Series(T_test), 'C3'),
    'cTc C3': agm.temperature_correction_clear(pd.Series(T_test), 'C3'),
    'cTo C4': agm.temperature_correction_overcast(pd.Series(T_test), 'C4'),
    'cTc C4': agm.temperature_correction_clear(pd.Series(T_test), 'C4'),
}).round(3)

,T (°C),cTo C3,cTc C3,cTo C4,cTc C4
0,12.5,0.722,0.889,0.113,0.000
1,15.0,0.902,0.961,0.696,0.392
2,20.0,1.218,1.087,1.236,1.590
3,25.0,1.418,1.167,1.448,2.121
4,30.0,1.435,1.174,1.516,2.290
5,35.0,1.198,1.079,1.438,2.096


> 🔎 The advantage of maize shows almost only in `cTc`: under an overcast sky,
> light limits both pathways alike. Below ~14 °C the C4 polynomials become
> negative and the functions return zero.

### 10.3 `gross_potential_productivity` — Gross potential productivity (GP)

> 📌 With `Ra` and `n/N` of September 15 and the mean temperature `Tmean`,
> how much dry matter would the standard crop (LAI = 5) produce per day, as
> soybean (C3) or maize (C4)?

In [ ]:
for pathway in ('C3', 'C4'):
    gp = agm.gross_potential_productivity(
        Ra, nN,
        cTc=agm.temperature_correction_clear(Tmean, pathway),
        cTo=agm.temperature_correction_overcast(Tmean, pathway),
    )
    print(f"GP ({pathway}) = {gp:.1f} kg DM ha-1 day-1")

GP (C3) = 365.0 kg DM ha-1 day-1
GP (C4) = 511.5 kg DM ha-1 day-1


### 10.4 `lai_correction` and `respiration_correction`

> 📌 Soybean has a maximum LAI of 4.5 (not 5, as the standard crop). And
> maintenance respiration costs more when the temperature is above 20 °C.

In [ ]:
print(f"LAI correction (LAI = 4.5) = {agm.lai_correction(4.5):.3f}")
print(f"LAI correction (LAI = 5.0) = {agm.lai_correction(5.0):.3f}")
print(f"CR at 18 °C = {agm.respiration_correction(18)}  |  CR at 24 °C = {agm.respiration_correction(24)}")

LAI correction (LAI = 4.5) = 0.487
LAI correction (LAI = 5.0) = 0.500
CR at 18 °C = 0.6  |  CR at 24 °C = 0.5


### 10.5 `potential_yield` — Yp accumulated over the cycle

> 📌 The farm's soybean was sown on **2023-11-05**, with a **125-day** cycle.
> With the mean temperature and solar radiation of each dekad of the season,
> compute the potential yield (Yp), accumulating dekad by dekad.

- `interval` sets the time step: `'d'` (daily), `'dekad'` (10-day periods)
  or `'M'` (monthly).
- With `sowing` and `cycle`, the first and last dekads only count the days
  with the crop in the field (column `ND`).
- Without an `Ra` column, it is computed from `lat` and the dates.

In [ ]:
dates = pd.date_range('2023-11-01', '2024-03-01', freq='D')
dates = dates[dates.day.isin([1, 11, 21])]          # start of each dekad

df_season = pd.DataFrame({
    'date':  dates,
    'Tmean': [19.8, 20.6, 21.0, 21.4, 21.9, 22.3, 22.6, 23.1, 23.4, 22.9, 22.5, 22.0, 21.6],
    'Rs':    [20.1, 21.4, 22.0, 21.2, 22.8, 21.5, 23.6, 24.1, 23.8, 22.0, 20.9, 20.2, 19.5],
    'P':     [48.0, 62.5, 35.2, 55.8, 41.0, 70.3, 12.4, 5.8, 22.1, 60.2, 45.7, 38.4, 30.0],
    'ETo':   [41.2, 43.8, 45.1, 44.0, 47.5, 45.3, 50.1, 51.6, 55.0, 46.2, 42.8, 40.5, 38.0],
})

yp = agm.potential_yield(df_season, lat=lat, pathway='C3', LAI=4.5, HI=0.35, moisture=13,
                         interval='dekad', sowing='2023-11-05', cycle=125)
Yp = yp.attrs['Yp']
print(f"Yp = {Yp:,.0f} kg/ha  ({Yp / 1000:.2f} t/ha)")
yp[['date', 'ND', 'Tmean', 'Ra', 'nN', 'cTo', 'cTc', 'GP', 'CR', 'Yp_period', 'Yp_cum']].round(2)

Yp = 5,276 kg/ha  (5.28 t/ha)

,date,ND,Tmean,Ra,nN,cTo,cTc,GP,CR,Yp_period,Yp_cum
0,2023-11-01,6,19.8,41.17,0.43,1.21,1.08,385.58,0.6,272.19,272.19
1,2023-11-11,10,20.6,41.85,0.48,1.25,1.10,409.14,0.5,401.14,673.34
2,2023-11-21,10,21.0,42.50,0.49,1.27,1.11,421.21,0.5,412.98,1086.31
3,2023-12-01,10,21.4,42.93,0.44,1.29,1.12,419.79,0.5,411.58,1497.90
4,2023-12-11,10,21.9,43.16,0.51,1.31,1.12,440.11,0.5,431.51,1929.40
5,2023-12-21,11,22.3,43.19,0.45,1.33,1.13,432.53,0.5,466.48,2395.88
6,2024-01-01,10,22.6,42.99,0.55,1.34,1.14,453.17,0.5,444.32,2840.20
7,2024-01-11,10,23.1,42.59,0.58,1.36,1.14,460.32,0.5,451.32,3291.52
8,2024-01-21,11,23.4,41.91,0.59,1.37,1.15,457.49,0.5,493.40,3784.92
9,2024-02-01,10,22.9,40.98,0.53,1.35,1.14,434.02,0.5,425.54,4210.46


### 10.6 `attainable_yield` — Yw by phase or in a single step

> 📌 There was a dry spell in December/January. Build the crop water balance
> by dekad (Kc and AWC of each phase) with `crop_water_balance` and penalize
> Yp for the water deficit.

The relation of Doorenbos & Kassam is $1 - Y_w/Y_p = k_y\,(1 - ETa/ETc)$. The
function offers both forms, chosen with `method`:

- **`'product'`** (default): one `ky` per phase, applied in sequence — the
  yield at the end of one phase is the starting point of the next (Battisti
  et al., 2013). With `phase=`, ETa and ETc of the dekads are summed per
  phase.
- **`'single'`**: a single whole-cycle `ky`, with the sums of ETa and ETc.

In [ ]:
import numpy as np

# Soybean parameters per phase (last day after sowing, Kc, root depth and ky)
soy_phases = {'Vegetative':      dict(end=50,  Kc=0.62, Zr=0.40, ky=0.2),
              'Flowering':       dict(end=70,  Kc=1.15, Zr=0.60, ky=0.8),
              'Yield formation': dict(end=105, Kc=1.15, Zr=0.60, ky=1.0),
              'Ripening':        dict(end=125, Kc=0.83, Zr=0.60, ky=0.0)}

# Phase of each dekad from its central day (days after sowing)
sow = pd.Timestamp('2023-11-05')
field_start = yp['date'].where(yp['date'] > sow, sow)
yp['DAS_mid'] = (field_start - sow).dt.days + 1 + (yp['ND'] - 1) / 2
limits = [p['end'] for p in soy_phases.values()]
yp['phase'] = [list(soy_phases)[i] for i in np.searchsorted(limits, yp['DAS_mid'])]

AW = 1.0                                              # available water of the soil (mm/cm)
yp['ETc'] = yp['phase'].map(lambda f: soy_phases[f]['Kc']) * yp['ETo'] * yp['ND'] / 10
yp['AWC'] = yp['phase'].map(lambda f: soy_phases[f]['Zr']) * 100 * AW
yp['P_cycle'] = yp['P'] * yp['ND'] / 10               # partial dekads at both ends

wb = agm.crop_water_balance(yp[['P_cycle', 'ETc', 'AWC']].rename(columns={'P_cycle': 'P'}))

# By phase (product)
yw = agm.attainable_yield(Yp, wb['ETa'], wb['ETc'],
                          ky={f: p['ky'] for f, p in soy_phases.items()},
                          phase=yp['phase'])
# Single step (whole-cycle ky of soybean = 0.85)
yw_single = agm.attainable_yield(Yp, wb['ETa'], wb['ETc'], ky=0.85, method='single')

print(f"Yp                      = {Yp:7.0f} kg/ha")
print(f"Yw by phase (product)   = {yw.attrs['Yw']:7.0f} kg/ha  →  CE = {yw.attrs['CE']:.3f}")
print(f"Yw in a single step     = {yw_single.attrs['Yw']:7.0f} kg/ha  →  CE = {yw_single.attrs['CE']:.3f}")
yw.round(3)

Yp                      =    5276 kg/ha
Yw by phase (product)   =    3152 kg/ha  →  CE = 0.597
Yw in a single step     =    4486 kg/ha  →  CE = 0.850


,ETc,ETa,ETa/ETc,ky,factor,Yw_start,Yw_end,loss,CE
phase,,,,,,,,,
Vegetative,127.174,127.174,1.000,0.2,1.000,5275.652,5275.652,0.000,1.000
Flowering,114.920,101.464,0.883,0.8,0.906,5275.652,4781.482,494.170,0.906
Yield formation,231.265,152.452,0.659,1.0,0.659,4781.482,3152.001,1629.481,0.597
Ripening,55.485,54.581,0.984,0.0,1.000,3152.001,3152.001,0.000,0.597


> 🔎 The product penalizes more because the deficit fell during **yield
> formation** (ky = 1.0), while the single step dilutes it over the whole
> cycle with the average ky of 0.85. The `CE` column shows the climatic
> efficiency accumulated phase by phase, and `loss` how much each phase took
> from the yield.

---
## 🎉 End of the tour

You have just run every public function of `agrometeorologiapy`, from start
to finish — from solar radiation to attainable yield — with a single,
consistent scenario.

- 📦 PyPI: <https://pypi.org/project/agrometeorologiapy/>
- 💻 Source code: <https://github.com/fcoliveira-utfpr/agrometeorologiapy>
- 📐 Formulas: [docs/FORMULAS.en.md](https://github.com/fcoliveira-utfpr/agrometeorologiapy/blob/main/docs/FORMULAS.en.md)
- 🐛 Found a problem? Open an *issue*: <https://github.com/fcoliveira-utfpr/agrometeorologiapy/issues>

```bash
pip install agrometeorologiapy
```

License BSD-3-Clause.